# Kathmandu Airbnb ML Analysis — Project Report Notebook

**Purpose:** Briefly document the project, methods, outcomes, and visual analyses using the repository's local processed datasets and report files. This notebook is designed to run from the repository root (or its `notebooks/` folder) and creates charts from available files.

> **Data caveat:** This is an academic/internship demonstration. Property/price data include synthetic sample data; the cancellation benchmark is from Portugal hotel bookings; Nepal tourist arrivals are national context, not Kathmandu occupancy. Results are not live market evidence.


## 1. Project Summary

The project explores accommodation-listing analysis, recommendation, nightly-price estimation, and dashboard-based insights for Kathmandu. Its Streamlit dashboard includes Executive Analytics, Property Recommendations, Price Prediction, Data Analysis, and Model Evaluation workspaces.

### Main objectives
- Explore listing characteristics, price distributions, neighbourhoods, and property types.
- Build a preference-based accommodation recommendation workflow.
- Train and evaluate a nightly-price regression model.
- Inspect and preprocess an external hotel cancellation benchmark for exploratory work.
- Present results and visual analyses through an interactive dashboard and reproducible reports.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed'
REPORTS = ROOT / 'reports'
print('Project root:', ROOT)
print('Processed data directory exists:', DATA.exists())
print('Reports directory exists:', REPORTS.exists())


## 2. Project Workflow

The following conceptual diagram summarizes the project pipeline from datasets through preparation and modelling to the dashboard/reporting layer.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.axis('off')
steps = [
    ('Input data', 'Listings • prices • availability\nTourism context • benchmark data'),
    ('Preparation', 'Audit • clean • feature engineering\nSeparate source and synthetic data'),
    ('ML / Analytics', 'Recommendation ranking\nGradient Boosting price regression\nBenchmark exploration'),
    ('Evaluation', 'Test metrics • CV • error analysis\nScenario-based recommendation checks'),
    ('Presentation', 'Streamlit dashboard\nReports • charts • notebook'),
]
xs = [0.10, 0.30, 0.50, 0.70, 0.90]
for i, (title, desc) in enumerate(steps):
    ax.text(xs[i], 0.55, title, ha='center', va='center', fontsize=11, weight='bold',
            bbox=dict(boxstyle='round,pad=0.7', facecolor='#E8F2FA', edgecolor='#3977A8'))
    ax.text(xs[i], 0.30, desc, ha='center', va='center', fontsize=9)
    if i < len(steps)-1:
        ax.annotate('', xy=(xs[i+1]-0.07, 0.55), xytext=(xs[i]+0.07, 0.55),
                   arrowprops=dict(arrowstyle='->', lw=1.8, color='#3977A8'))
ax.set_title('Project Workflow Overview', fontsize=15, weight='bold')
plt.tight_layout()
plt.show()


## 3. Data Inventory and Basic Audit

Load the available processed datasets. Missing files are skipped with a notice so the notebook can still run when a checkout contains only a subset of artifacts.


In [ ]:
candidate_files = {
    'properties': 'properties_clean.csv',
    'price_observations': 'price_observations_clean.csv',
    'availability': 'availability_clean.csv',
    'cancellations': 'booking_cancellations_clean.csv',
    'tourist_arrivals': 'nepal_monthly_tourist_arrivals_clean.csv',
    'recommendation_features': 'recommendation_features.csv',
    'price_features': 'price_prediction_features.csv',
}
data = {}
for key, filename in candidate_files.items():
    path = DATA / filename
    if path.exists():
        try:
            data[key] = pd.read_csv(path)
            print(f'{key}: {data[key].shape}')
        except Exception as exc:
            print(f'{key}: could not read ({exc})')
    else:
        print(f'{key}: file not found — skipped')

inventory = pd.DataFrame([{'dataset': k, 'rows': len(v), 'columns': len(v.columns)} for k,v in data.items()])
display(inventory)


In [ ]:
if 'properties' in data:
    properties = data['properties']
    print('Property data shape:', properties.shape)
    print('Columns:')
    display(pd.DataFrame({'column': properties.columns, 'dtype': properties.dtypes.astype(str).values}))
    if 'published' in properties:
        print('Published records:', int(properties['published'].fillna(False).astype(bool).sum()))
    elif 'is_published' in properties:
        print('Published records:', int(properties['is_published'].fillna(False).astype(bool).sum()))
else:
    print('Property dataset unavailable; audit skipped.')


## 4. Executive Analytics — Listing and Market Context

These plots recreate common dashboard-style views from the processed property dataset when the required columns are available.


In [ ]:
if 'properties' in data:
    df = data['properties'].copy()
    price_col = next((c for c in ['nightly_price_npr','price_npr','base_price_npr','price'] if c in df.columns), None)
    if price_col:
        prices = pd.to_numeric(df[price_col], errors='coerce').dropna()
        prices = prices[prices > 0]
        if not prices.empty:
            fig, ax = plt.subplots(figsize=(9, 4.5))
            ax.hist(prices, bins=30, edgecolor='white')
            ax.set_title('Nightly Price Distribution (Available Sample)')
            ax.set_xlabel('Price (NPR)'); ax.set_ylabel('Listing / observation count')
            plt.tight_layout(); plt.show()
            print(f'Valid price records: {len(prices):,}; median: NPR {prices.median():,.2f}; mean: NPR {prices.mean():,.2f}')
        else:
            print('No valid positive price values found.')
    else:
        print('Price column not found; available columns:', list(df.columns))
else:
    print('Properties data unavailable; chart skipped.')


In [ ]:
if 'properties' in data:
    df = data['properties'].copy()
    neighborhood_col = next((c for c in ['neighbourhood','neighborhood','area'] if c in df.columns), None)
    price_col = next((c for c in ['nightly_price_npr','price_npr','base_price_npr','price'] if c in df.columns), None)
    if neighborhood_col:
        counts = df[neighborhood_col].fillna('Unknown').value_counts().head(15).sort_values()
        fig, ax = plt.subplots(figsize=(10, 5))
        counts.plot(kind='barh', ax=ax)
        ax.set_title('Top Neighbourhoods by Listing Count'); ax.set_xlabel('Listings / records')
        plt.tight_layout(); plt.show()
    if neighborhood_col and price_col:
        tmp = df.assign(_price=pd.to_numeric(df[price_col], errors='coerce')).dropna(subset=['_price'])
        medians = tmp.groupby(neighborhood_col)['_price'].median().sort_values(ascending=False).head(15).sort_values()
        if len(medians):
            fig, ax = plt.subplots(figsize=(10, 5)); medians.plot(kind='barh', ax=ax)
            ax.set_title('Median Sample Nightly Price by Neighbourhood'); ax.set_xlabel('Median price (NPR)')
            plt.tight_layout(); plt.show()
    else:
        print('Neighbourhood or price columns unavailable for neighbourhood charts.')


In [ ]:
if 'properties' in data:
    df = data['properties']
    type_col = next((c for c in ['property_type','accommodation_type','listing_type'] if c in df.columns), None)
    if type_col:
        counts = df[type_col].fillna('Unknown').value_counts().head(12).sort_values()
        fig, ax = plt.subplots(figsize=(9, 4.5)); counts.plot(kind='barh', ax=ax)
        ax.set_title('Property Type Composition'); ax.set_xlabel('Records')
        plt.tight_layout(); plt.show()
    else:
        print('Property-type field not found.')


## 5. Tourism Context

National tourist arrivals are included as contextual information only. They should not be interpreted as Kathmandu occupancy or Airbnb booking demand.


In [ ]:
if 'tourist_arrivals' in data:
    tourism = data['tourist_arrivals'].copy()
    date_col = next((c for c in ['month_date','date','month'] if c in tourism.columns), None)
    value_col = next((c for c in ['tourist_arrivals','arrivals','visitor_count','total_arrivals'] if c in tourism.columns), None)
    if date_col and value_col:
        tourism[date_col] = pd.to_datetime(tourism[date_col], errors='coerce')
        tourism[value_col] = pd.to_numeric(tourism[value_col], errors='coerce')
        tourism = tourism.dropna(subset=[date_col,value_col]).sort_values(date_col)
        if not tourism.empty:
            fig, ax = plt.subplots(figsize=(10,4)); ax.plot(tourism[date_col], tourism[value_col], marker='o')
            ax.set_title('Nepal Monthly Tourist Arrivals (Context)'); ax.set_ylabel('Arrivals'); ax.set_xlabel('Month')
            plt.xticks(rotation=45); plt.tight_layout(); plt.show()
            display(tourism[[date_col,value_col]].tail(12))
    else:
        print('Tourism dataset fields not recognized. Columns:', list(tourism.columns))
else:
    print('Tourism context dataset unavailable; skipped.')


## 6. Recommendation System — Brief Report

The recommender combines structured preference signals with text similarity and ranking logic. Inputs described in the project include budget compatibility, neighbourhood, property/room type, guest capacity, bedrooms, bathrooms, amenities, ratings, review counts, and listing descriptions. TF-IDF and cosine similarity are used for text-based similarity.

**Outcome interpretation:** scenario-based evaluation and weight comparisons are heuristic preference-match diagnostics. They are not verified ground-truth recommendation accuracy, precision, recall, or user satisfaction. The notebook summarizes the approach; charts below are created only if recommendation report files are present.


In [ ]:
rec_report_candidates = [REPORTS / 'recommendation_evaluation.csv', REPORTS / 'recommendation_quality_report.csv', REPORTS / 'recommendation_scenario_evaluation.csv']
rec_report = next((pd.read_csv(p) for p in rec_report_candidates if p.exists()), None)
if rec_report is not None:
    display(rec_report.head(20))
    numeric = rec_report.select_dtypes(include='number')
    if not numeric.empty:
        metric = numeric.columns[0]
        fig, ax = plt.subplots(figsize=(9,4)); numeric[metric].plot(kind='bar', ax=ax)
        ax.set_title(f'Recommendation Report: {metric}'); ax.set_ylabel(metric)
        plt.tight_layout(); plt.show()
else:
    print('No recognized recommendation evaluation report found; chart skipped. Check reports/ for available files.')


## 7. Price Prediction — Methods and Reported Outcomes

The current reported model is a Gradient Boosting Regressor. Its feature pipeline includes listing attributes, location/type, capacity, bedrooms/bathrooms, amenities, and stay-related features. The report describes a property-level train/test split to reduce overlap of property IDs.

**Reported test metrics (from project README):**

| Metric | Reported result |
|---|---:|
| MAE | NPR 148.33 |
| RMSE | NPR 218.93 |
| R² | 0.9942 |
| Baseline MAE | NPR 158.85 |
| MAE improvement | 6.62% |

Reported split: 808 training observations from 320 properties; 192 test observations from 80 properties; zero property IDs shared across train and test. These figures are based on prepared sample data and are not verified real-market accuracy. The model also uses the existing baseline nightly price as an input, so the high scores should not be interpreted as independent price discovery.


In [ ]:
metric_path = REPORTS / 'price_model_metrics.csv'
if metric_path.exists():
    metrics_df = pd.read_csv(metric_path); display(metrics_df)
    # Plot metric values when a recognizable metric/value schema is present.
    value_col = next((c for c in metrics_df.columns if c.lower() in ['value','metric_value','score']), None)
    name_col = next((c for c in metrics_df.columns if c.lower() in ['metric','name','metric_name']), None)
    if value_col and name_col:
        plot_df = metrics_df.dropna(subset=[value_col]).set_index(name_col)[value_col]
        fig, ax = plt.subplots(figsize=(8,4)); plot_df.plot(kind='bar', ax=ax)
        ax.set_title('Saved Price Model Metrics'); plt.tight_layout(); plt.show()
else:
    print('Saved price_model_metrics.csv not found; displaying README-reported outcomes above.')


In [ ]:
comparison_path = REPORTS / 'price_model_comparison.csv'
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path); display(comparison)
    for metric in ['MAE','RMSE','R2','R²']:
        if metric in comparison.columns:
            fig, ax = plt.subplots(figsize=(8,4)); comparison.set_index('model')[metric].plot(kind='bar', ax=ax)
            ax.set_title(f'Model Comparison — {metric}'); plt.tight_layout(); plt.show()
else:
    print('Model comparison report not found; skipped.')


In [ ]:
cv_path = REPORTS / 'price_model_cross_validation.csv'
if cv_path.exists():
    cv = pd.read_csv(cv_path); display(cv)
    if 'fold' in cv.columns:
        for metric in ['GB_MAE','GB_RMSE','GB_R2']:
            if metric in cv.columns:
                fig, ax = plt.subplots(figsize=(8,4)); cv.set_index('fold')[metric].plot(marker='o', ax=ax)
                ax.set_title(f'Gradient Boosting Cross-Validation — {metric}'); plt.tight_layout(); plt.show()
else:
    print('Cross-validation report not found; skipped.')


## 8. Booking Cancellation Benchmark — Exploratory Result

The cancellation workflow uses the Hotel Booking Demand dataset from Portugal. It is a separate benchmark and does not represent Kathmandu Airbnb booking records. The repository reports 119,390 original rows and 25 columns, an exploratory cancellation rate near 37.04%, and 119,209 rows after preprocessing checks. Treat this as benchmark data inspection/preprocessing, not a Kathmandu cancellation model.


In [ ]:
if 'cancellations' in data:
    cancel = data['cancellations'].copy()
    target = next((c for c in ['is_canceled','is_cancelled','canceled'] if c in cancel.columns), None)
    if target:
        rates = cancel[target].value_counts(normalize=True).sort_index()*100
        display(rates.rename('percent').to_frame())
        fig, ax = plt.subplots(figsize=(6,4)); rates.plot(kind='bar', ax=ax)
        ax.set_title('Cancellation Benchmark Target Distribution'); ax.set_ylabel('Percent')
        plt.tight_layout(); plt.show()
    else:
        print('Cancellation target column not found; dataset columns:', list(cancel.columns))
else:
    print('Processed cancellation benchmark unavailable; summary above is from the project README.')


## 9. Overall Outcomes and Limitations

### Outcomes
- A multi-workspace Streamlit dashboard was implemented for analytics, recommendation, price prediction, data analysis, and model evaluation.
- A property recommendation workflow supports personalized ranking and similar-property discovery.
- A Gradient Boosting nightly-price model was trained and evaluated with reported metrics and property-level splitting.
- Data preparation, evaluation reports, and benchmark cancellation preprocessing are included.
- Charts in this notebook provide a reproducible, static report view based on the local repository artifacts.

### Limitations
- Property and pricing samples are synthetic; listings are not verified live Airbnb inventory.
- External cancellation benchmark is Portugal hotel data, not Kathmandu accommodation.
- National tourist-arrival figures are not local occupancy or booking demand.
- Recommendation metrics are scenario-based heuristics, not user-ground-truth evaluation.
- Price model performance is limited by the sample data and the use of baseline price as a feature.
- No operational reservations, live availability, or verified market pricing are provided.


## 10. Conclusion

This project demonstrates an end-to-end machine-learning and analytics workflow for accommodation data: data preparation, recommendation, price estimation, evaluation, and dashboard presentation. It is a useful academic prototype and technical demonstration. Further validation requires permitted, verified Kathmandu listing and pricing observations, local occupancy/booking data, and user-based recommendation feedback.

**Notebook usage:** Run cells from top to bottom. Charts are generated from files available in `data/processed/` and `reports/`. Missing artifacts are reported and their charts skipped. The workflow does not fabricate graph values when files are missing.
